# Pydantic

它通过在运行时强制执行类型提示，确保数据的正确性和一致性，是 生产场景首选 。
## 基本使用
前提：
- 所有结构化输出的数据模型必须继承`BaseModel`
- 使用`类型提示`
- 使用`Field()`添加字段默认值和描述

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool  # 从 core 导入，避免 langgraph 版本冲突
from rich import print as rprint
from langchain_qwq import ChatQwen
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

In [2]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    job: str = Field(description="职业")

model_with_structured_output = model.with_structured_output(Person)

result = model_with_structured_output.invoke('张三是一名30岁的前端开发工程师')
rprint(result) # Person(name='张三', age=30, job='前端开发工程师')
print(type(result)) # <class 'Person'>

Person(name='张三', age=30, job='前端开发工程师')

<class '__main__.Person'>


In [ ]:

class Movie(BaseModel):
    """
    电影信息
    """
    title: str = Field(description="电影名称")
    year: int = Field(description="上映年份")
    rating: float = Field(description="评分")
    description: str = Field(description="电影描述")

model_with_structured_output = model.with_structured_output(Movie)
result = model_with_structured_output.invoke("给我一个电影的信息")
rprint(result)
rprint(result.model_dump()) # 将模型转换为字典

Movie(
    title='盗梦空间',
    year=2010,
    rating=9.4,
    description='《盗梦空间》是由克里斯托弗·诺兰执导的科幻动作片，讲述了一群利用梦境窃取秘密的特工，被雇佣在目标潜
意识中植入一个想法的故事。影片以精妙的叙事结构、层层嵌套的梦境空间和深刻的情感内核闻名，被誉为21世纪最具影响力的科
幻电影之一。'
)

{
    'title': '盗梦空间',
    'year': 2010,
    'rating': 9.4,
    'description': 
'《盗梦空间》是由克里斯托弗·诺兰执导的科幻动作片，讲述了一群利用梦境窃取秘密的特工，被雇佣在目标潜意识中植入一个想
法的故事。影片以精妙的叙事结构、层层嵌套的梦境空间和深刻的情感内核闻名，被誉为21世纪最具影响力的科幻电影之一。'
}

### 可选字段
使用`Optional`修饰字段，表示该字段可以为空。

In [9]:
from typing import Optional

class Movie(BaseModel):
    title: str = Field(description="电影名称")
    year: int = Field(description="上映年份")
    rating: float = Field(description="评分")
    description: Optional[str] = Field(description="电影描述")

model_with_structured_output = model.with_structured_output(Movie)
result = model_with_structured_output.invoke("给我一个电影的信息，不要添加描述")
rprint(result.model_dump())


{'title': '肖申克的救赎', 'year': 1994, 'rating': 9.7, 'description': '暂无描述'}

### 默认值
LLM 未提供的信息会使用默认值。格式如下：
- `Field(default="默认值", description="描述")`

注意：不同模型提供商对default字段的支持是不同的

In [11]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(1,description="年龄")
    occupation: str = Field(description="职业")
structured_llm = model.with_structured_output(Person)
result = structured_llm.invoke("张三是一名医生")
rprint(result)

Person(name='张三', age=1, occupation='医生')

### 枚举类型

In [18]:
from enum import Enum

class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class Task(BaseModel):
    title: str
    priority: Priority # 只能是 LOW/MEDIUM/HIGH

class Status(str, Enum):
    ACTIVE = "激活"
    INACTIVE = "未激活"

class User(BaseModel):
    status: Status # 只能是 ACTIVE 或 INACTIVE

structured_llm = model.with_structured_output(User)

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

structured_llm = model.with_structured_output(CustomerInfo)
result = structured_llm.invoke("客户姓名是张三，电话号码是1234567890，邮箱是zhangsan@example.com，问题描述是电池电量不足，紧急程度是高")
rprint(result.model_dump()['urgency'])

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
rprint(result)

高

CustomerInfo(
    name='王小明',
    phone='138-1234-5678',
    email='null',
    issue='订单一直没发货',
    urgency=<Priority.HIGH: '高'>
)